# Problem: Implement Transformer Encoder-Decoder for Machine Translation

### Problem Statement
You are tasked with implementing a **Transformer-based Encoder-Decoder architecture** for neural machine translation using PyTorch. You will train the model on the **Multi30k dataset** (German-to-English translation) and evaluate its performance using the **BLEU score**.

### Requirements
1. **Data Acquisition**: Load and preprocess the Multi30k dataset (German → English).
2. **Data Wrangling**: Tokenize text, build vocabularies, and create data loaders.
3. **Model Implementation**: Implement a Transformer Encoder-Decoder from scratch or using PyTorch components.
4. **Training**: Train the model using appropriate loss function and optimizer.
5. **Evaluation**: Calculate BLEU score on the test set.

### Constraints
- Use the Multi30k dataset for German-English translation.
- Implement key Transformer components (multi-head attention, positional encoding, etc.).
- Use teacher forcing during training.
- Evaluate using BLEU score metric.

<details>
  <summary>💡 Hint</summary>
  Use torchtext for data loading. Implement positional encoding and multi-head attention mechanisms. Use CrossEntropyLoss and Adam optimizer. For BLEU, use nltk or torchtext metrics.
</details>

## Step 1: Data Acquisition

Load the **Multi30k dataset** using `torchtext` or download it directly. This dataset contains German-English parallel sentences for translation tasks.

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import math
import spacy
from collections import Counter
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np

# For torchtext compatibility
try:
    from torchtext.datasets import Multi30k
    from torchtext.data.utils import get_tokenizer
except:
    print("Using alternative data loading method")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load spacy tokenizers
try:
    spacy_de = spacy.load('de_core_news_sm')
    spacy_en = spacy.load('en_core_web_sm')
except:
    print("Spacy models not found. Using basic tokenizer.")
    spacy_de = None
    spacy_en = None

def tokenize_de(text):
    if spacy_de:
        return [tok.text for tok in spacy_de.tokenizer(text)]
    else:
        return text.lower().split()

def tokenize_en(text):
    if spacy_en:
        return [tok.text for tok in spacy_en.tokenizer(text)]
    else:
        return text.lower().split()

# Load Multi30k dataset
train_data, valid_data, test_data = Multi30k(split=('train', 'valid', 'test'), language_pair=('de', 'en'))

# Convert iterators to lists for reusability
train_data = list(train_data)
valid_data = list(valid_data)
test_data = list(test_data)

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(valid_data)}")
print(f"Test examples: {len(test_data)}")
print(f"\nExample: {train_data[0]}")

## Step 2: Data Wrangling

Tokenize the German and English sentences, build vocabularies, and create data loaders with proper padding and batching.

In [ ]:
# Special tokens
UNK_IDX, PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<sos>', '<eos>']

# Build vocabulary
def build_vocab(data, tokenizer, min_freq=2):
    counter = Counter()
    for src, tgt in data:
        counter.update(tokenizer(src))
    
    vocab = {symbol: idx for idx, symbol in enumerate(special_symbols)}
    idx = len(special_symbols)
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab

# Build vocabularies for German and English
src_vocab = build_vocab(train_data, tokenize_de)
tgt_vocab = build_vocab([(tgt, src) for src, tgt in train_data], tokenize_en)

print(f"German vocabulary size: {len(src_vocab)}")
print(f"English vocabulary size: {len(tgt_vocab)}")

# Create reverse vocabularies for decoding
src_idx_to_token = {idx: token for token, idx in src_vocab.items()}
tgt_idx_to_token = {idx: token for token, idx in tgt_vocab.items()}

In [ ]:
# Convert text to indices
def text_to_indices(text, vocab, tokenizer):
    return [vocab.get(token, UNK_IDX) for token in tokenizer(text)]

# Collate function for DataLoader
def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_indices = text_to_indices(src_sample, src_vocab, tokenize_de)
        tgt_indices = text_to_indices(tgt_sample, tgt_vocab, tokenize_en)
        
        src_batch.append(torch.tensor([SOS_IDX] + src_indices + [EOS_IDX]))
        tgt_batch.append(torch.tensor([SOS_IDX] + tgt_indices + [EOS_IDX]))
    
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_batch, tgt_batch

# Create data loaders
BATCH_SIZE = 32

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(valid_loader)}")
print(f"Number of test batches: {len(test_loader)}")

## Step 3: Model Implementation

Implement the Transformer Encoder-Decoder architecture with:
- Positional Encoding
- Multi-Head Attention
- Encoder and Decoder blocks
- Complete Seq2Seq Transformer model

In [ ]:
# Implement Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
# Implement Transformer Model
class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, nhead=8, 
                 num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=2048, dropout=0.1):
        super(Seq2SeqTransformer, self).__init__()
        
        self.d_model = d_model
        
        # Embeddings
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        # Output projection
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, 
                src_padding_mask=None, tgt_padding_mask=None, memory_key_padding_mask=None):
        # Embed and add positional encoding
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        
        # Transformer forward pass
        output = self.transformer(
            src_emb, tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        
        # Project to vocabulary
        return self.fc_out(output)
    
    def encode(self, src, src_mask):
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        return self.transformer.encoder(src_emb, src_mask)
    
    def decode(self, tgt, memory, tgt_mask):
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        return self.transformer.decoder(tgt_emb, memory, tgt_mask)

In [ ]:
# Helper functions for mask generation
def generate_square_subsequent_mask(sz, device):
    """Generate causal mask for decoder (prevents attending to future tokens)"""
    mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask

def create_mask(src, tgt, pad_idx, device):
    """Create all necessary masks for the transformer"""
    src_seq_len = src.shape[1]
    tgt_seq_len = tgt.shape[1]
    
    # Causal mask for target (decoder)
    tgt_mask = generate_square_subsequent_mask(tgt_seq_len, device)
    src_mask = None  # Encoder doesn't need causal mask
    
    # Padding masks
    src_padding_mask = (src == pad_idx)
    tgt_padding_mask = (tgt == pad_idx)
    
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

## Step 4: Training

Train the Transformer model using:
- **Loss Function**: CrossEntropyLoss (ignore padding index)
- **Optimizer**: Adam with learning rate scheduling
- **Teacher Forcing**: Use target sequences during training

In [ ]:
# Initialize model, loss, and optimizer
model = Seq2SeqTransformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Training function
def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    for batch_idx, (src, tgt) in enumerate(train_loader):
        src = src.to(device)
        tgt = tgt.to(device)
        
        # Target input and output (shifted by one position)
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        
        # Create masks
        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
            src, tgt_input, PAD_IDX, device
        )
        
        # Forward pass
        optimizer.zero_grad()
        logits = model(
            src, tgt_input,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        
        # Calculate loss
        loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_output.reshape(-1))
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % 100 == 0:
            print(f"  Batch {batch_idx + 1}/{len(train_loader)}, Loss: {loss.item():.4f}")
    
    return total_loss / len(train_loader)

# Evaluation function
def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for src, tgt in val_loader:
            src = src.to(device)
            tgt = tgt.to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            
            src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
                src, tgt_input, PAD_IDX, device
            )
            
            logits = model(
                src, tgt_input,
                src_mask=src_mask,
                tgt_mask=tgt_mask,
                src_padding_mask=src_padding_mask,
                tgt_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask
            )
            
            loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_output.reshape(-1))
            total_loss += loss.item()
    
    return total_loss / len(val_loader)

In [ ]:
# Train the model
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, valid_loader, criterion, device)
    
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")

print("\nTraining completed!")

## Step 5: Evaluation

Evaluate the trained model using **BLEU score** on the test set. Implement greedy decoding or beam search for translation.

In [ ]:
# Implement translation function with greedy decoding
def translate(model, src_sentence, src_vocab, tgt_vocab, device, max_len=50):
    model.eval()
    
    # Tokenize and convert to indices
    src_indices = text_to_indices(src_sentence, src_vocab, tokenize_de)
    src_tensor = torch.tensor([SOS_IDX] + src_indices + [EOS_IDX]).unsqueeze(0).to(device)
    
    # Encode source
    with torch.no_grad():
        src_mask = None
        src_padding_mask = (src_tensor == PAD_IDX)
        
        src_emb = model.pos_encoder(model.src_embedding(src_tensor) * math.sqrt(model.d_model))
        memory = model.transformer.encoder(src_emb, src_mask, src_padding_mask)
        
        # Decode with greedy search
        tgt_indices = [SOS_IDX]
        
        for i in range(max_len):
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            tgt_mask = generate_square_subsequent_mask(len(tgt_indices), device)
            
            tgt_emb = model.pos_encoder(model.tgt_embedding(tgt_tensor) * math.sqrt(model.d_model))
            output = model.transformer.decoder(tgt_emb, memory, tgt_mask)
            logits = model.fc_out(output)
            
            next_token = logits[0, -1].argmax().item()
            tgt_indices.append(next_token)
            
            if next_token == EOS_IDX:
                break
    
    # Convert indices back to words
    tgt_tokens = [tgt_idx_to_token.get(idx, '<unk>') for idx in tgt_indices[1:-1]]
    return ' '.join(tgt_tokens)

In [ ]:
# Calculate BLEU score
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

def calculate_bleu(model, test_data, src_vocab, tgt_vocab, device, max_samples=1000):
    model.eval()
    references = []
    hypotheses = []
    
    smoothing = SmoothingFunction().method1
    
    for i, (src, tgt) in enumerate(test_data[:max_samples]):
        if i % 100 == 0:
            print(f"Translating {i}/{max_samples}...")
        
        # Get translation
        translation = translate(model, src, src_vocab, tgt_vocab, device)
        
        # Reference (ground truth)
        ref_tokens = tokenize_en(tgt)
        hyp_tokens = translation.split()
        
        references.append([ref_tokens])
        hypotheses.append(hyp_tokens)
    
    # Calculate corpus BLEU
    bleu_score = corpus_bleu(references, hypotheses, smoothing_function=smoothing)
    return bleu_score * 100  # Convert to percentage

# Evaluate BLEU score
bleu_score = calculate_bleu(model, test_data, src_vocab, tgt_vocab, device, max_samples=500)
print(f"\nBLEU Score: {bleu_score:.2f}")

In [ ]:
# Example translations
print("\n" + "="*80)
print("Sample Translations")
print("="*80)

for i in range(10):
    src_sentence, tgt_sentence = test_data[i]
    translation = translate(model, src_sentence, src_vocab, tgt_vocab, device)
    
    print(f"\nExample {i+1}:")
    print(f"Source (DE): {src_sentence}")
    print(f"Target (EN): {tgt_sentence}")
    print(f"Translation: {translation}")
    print("-" * 80)